In [1]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"]= "HF_TOKEN_REMOVED"

In [9]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFaceEmbeddings
from dotenv import load_dotenv
load_dotenv()

llm= HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.2-1B-Instruct",
    task="conversational"
)
model= ChatHuggingFace(llm= llm)

from langchain.vectorstores import Chroma

In [10]:
from langchain_core.documents import Document

In [11]:
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [12]:
docs= [doc1, doc2, doc3, doc4, doc5]

In [13]:
vector_store = Chroma(
    embedding_function= HuggingFaceEmbeddings(),
    persist_directory='chroma_db',
    collection_name='sample'
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6704.42it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# add documents
vector_store.add_documents(docs)

['c521a6e0-1822-452b-94f5-26f398aca6fa',
 '3f08559e-2232-48e6-87f0-cb977346e679',
 '73338078-0164-42d3-a64e-6f64e93c9e45',
 '39d0ef25-518a-429a-ac31-b7de8dbd9d5c',
 'cbce14b6-d9f0-4ade-83cc-b9c4188152cb']

In [15]:
# view documents
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['c521a6e0-1822-452b-94f5-26f398aca6fa',
  '3f08559e-2232-48e6-87f0-cb977346e679',
  '73338078-0164-42d3-a64e-6f64e93c9e45',
  '39d0ef25-518a-429a-ac31-b7de8dbd9d5c',
  'cbce14b6-d9f0-4ade-83cc-b9c4188152cb'],
 'embeddings': array([[-0.03964757, -0.00801652, -0.01412725, ...,  0.06814158,
         -0.00116092, -0.02425425],
        [-0.01992601,  0.00161879, -0.00959344, ...,  0.02433111,
         -0.003414  , -0.01722026],
        [-0.04256174,  0.01060317, -0.01305763, ...,  0.04738364,
         -0.017487  ,  0.00204662],
        [-0.02734392, -0.03300869,  0.01565379, ...,  0.01462708,
          0.01177986,  0.00556421],
        [ 0.02301699, -0.02011731, -0.00123577, ...,  0.02533478,
          0.00280531, -0.05120282]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [16]:
# Search documents
vector_store.similarity_search(
    query= 'Who is a bowler?',
    k= 2 # for number of outputs
)

[Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [17]:
# Search documents with score
vector_store.similarity_search_with_score(
    query= 'Who is a bowler?',
    k= 2 # for number of outputs
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.1457648277282715),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  1.2282029390335083)]

In [18]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8879570960998535),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  2.0017189979553223)]

In [19]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='c521a6e0-1822-452b-94f5-26f398aca6fa', document=updated_doc1)

In [20]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['c521a6e0-1822-452b-94f5-26f398aca6fa',
  '3f08559e-2232-48e6-87f0-cb977346e679',
  '73338078-0164-42d3-a64e-6f64e93c9e45',
  '39d0ef25-518a-429a-ac31-b7de8dbd9d5c',
  'cbce14b6-d9f0-4ade-83cc-b9c4188152cb'],
 'embeddings': array([[-0.04108332, -0.0252802 , -0.02145072, ...,  0.02231622,
         -0.02287274, -0.00913   ],
        [-0.01992601,  0.00161879, -0.00959344, ...,  0.02433111,
         -0.003414  , -0.01722026],
        [-0.04256174,  0.01060317, -0.01305763, ...,  0.04738364,
         -0.017487  ,  0.00204662],
        [-0.02734392, -0.03300869,  0.01565379, ...,  0.01462708,
          0.01177986,  0.00556421],
        [ 0.02301699, -0.02011731, -0.00123577, ...,  0.02533478,
          0.00280531, -0.05120282]], shape=(5, 768)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple ce

In [22]:
# delete document
vector_store.delete(ids=['cbce14b6-d9f0-4ade-83cc-b9c4188152cb'])

In [23]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['c521a6e0-1822-452b-94f5-26f398aca6fa',
  '3f08559e-2232-48e6-87f0-cb977346e679',
  '73338078-0164-42d3-a64e-6f64e93c9e45',
  '39d0ef25-518a-429a-ac31-b7de8dbd9d5c'],
 'embeddings': array([[-0.04108332, -0.0252802 , -0.02145072, ...,  0.02231622,
         -0.02287274, -0.00913   ],
        [-0.01992601,  0.00161879, -0.00959344, ...,  0.02433111,
         -0.003414  , -0.01722026],
        [-0.04256174,  0.01060317, -0.01305763, ...,  0.04738364,
         -0.017487  ,  0.00204662],
        [-0.02734392, -0.03300869,  0.01565379, ...,  0.01462708,
          0.01177986,  0.00556421]], shape=(4, 768)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league